**Training Script**

In [50]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="/corpus.txt",
    model_prefix="hy_bpe",
    vocab_size=300,
    model_type="bpe",
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load("hy_bpe.model")

vocab_size = sp.get_piece_size()
print("Total number of tokens in the vocabulary:", vocab_size)

vocab = [sp.id_to_piece(i) for i in range(vocab_size)]

print("\nFirst 30 vocabulary entries:")
for token in vocab[:30]:
    print(token)

print("\nLast 30 vocabulary entries:")
for token in vocab[-30:]:
    print(token)

def observe(clean_vocab):
    single_chars, subwords, full_words = [], [], []

    for token in clean_vocab:
        if token == "":
            continue

        length = len(token)
        if length == 1:
            single_chars.append(token)
        elif 2 <= length <= 4:
            subwords.append(token)
        else:
            full_words.append(token)
    print("\nOBSERVATION:\n")
    print("Single characters:", len(single_chars))
    print("Subword fragments (2–4 chars):", len(subwords))
    print("Full words (5+ chars):", len(full_words))

    print("\nExamples of single characters:", single_chars[:20])
    print("Examples of subwords:", subwords[:20])
    print("Examples of full words:", full_words[:20])

clean_vocab = [v.replace("▁", "") for v in vocab]

observe(clean_vocab)



Total number of tokens in the vocabulary: 300

First 30 vocabulary entries:
<unk>
<s>
</s>
ու
ան
այ
եր
ար
ուն
▁հ
ում
ակ
ութ
▁է
ությ
են
ություն
▁Հ
ներ
աս
▁Հայ
▁կ
որ
ամ
ական
եւ
ատ
▁են
▁մ
▁հայ

Last 30 vocabulary entries:
Բ
չ
ջ
փ
Կ
Ն
Տ
ձ
Ե
Մ
Գ
Դ
Ծ
Պ
օ
Թ
Լ
Խ
Շ
Ռ
Ս
Վ
Ֆ
,
Ը
Ի
Ձ
Ղ
Ո
Ջ

OBSERVATION:

Single characters: 100
Subword fragments (2–4 chars): 158
Full words (5+ chars): 41

Examples of single characters: ['հ', 'է', 'Հ', 'կ', 'մ', 'ա', 'լ', 'շ', 'գ', 'դ', 'Ա', 'բ', 'ժ', 'Բ', 'տ', 'Կ', 'Ն', 'Տ', 'զ', 'պ']
Examples of subwords: ['<s>', '</s>', 'ու', 'ան', 'այ', 'եր', 'ար', 'ուն', 'ում', 'ակ', 'ութ', 'ությ', 'են', 'ներ', 'աս', 'Հայ', 'որ', 'ամ', 'ական', 'եւ']
Examples of full words: ['<unk>', 'ություն', 'ությունը', 'աստան', 'Հայաստան', 'կարեւոր', 'Հայաստանում', 'կական', 'ության', 'ավանդ', 'ունեն', 'աշխարհ', 'ությունն', 'ժողով', 'ավանդական', 'անում', 'իրում', 'ակակից', 'ժաման', 'համար']


**Encoding and Decoding Script**

In [25]:
sentences = {
    "S1": "Հայաստանն ունի հարուստ պատմություն։",
    "S2": "Արհեստական բանականությունը արագ զարգանում է։",
    "S3": "Ծրագրավորումը կարևոր հմտություն է ապագայի համար։"
}

for key, sentence in sentences.items():
    print(f"\n{key}: {sentence}")

    pieces = sp.encode(sentence, out_type=str)
    print("Token pieces:", pieces)

    ids = sp.encode(sentence, out_type=int)
    print("Token IDs:", ids)

    decoded = sp.decode(ids)
    print("Decoded:", decoded)


S1: Հայաստանն ունի հարուստ պատմություն։
Token pieces: ['▁Հայաստան', 'ն', '▁ունի', '▁հարուստ', '▁պ', 'ատ', 'մ', 'ություն', '։']
Token IDs: [35, 236, 60, 221, 95, 26, 242, 16, 246]
Decoded: Հայաստանն ունի հարուստ պատմություն։

S2: Արհեստական բանականությունը արագ զարգանում է։
Token pieces: ['▁Ար', 'հ', 'եստ', 'ական', '▁բ', 'ան', 'ականությունը', '▁արագ', '▁զարգ', 'անում', '▁է', '։']
Token IDs: [149, 247, 98, 24, 56, 4, 229, 161, 132, 158, 13, 246]
Decoded: Արհեստական բանականությունը արագ զարգանում է։

S3: Ծրագրավորումը կարևոր հմտություն է ապագայի համար։
Token pieces: ['▁Ծ', 'րագ', 'րա', 'վ', 'որ', 'ումը', '▁կարեւոր', '▁հ', 'մ', 'տ', 'ություն', '▁է', '▁ապագայի', '▁համար', '։']
Token IDs: [188, 202, 73, 252, 22, 208, 40, 9, 242, 244, 16, 13, 220, 165, 246]
Decoded: Ծրագրավորումը կարեւոր հմտություն է ապագայի համար։


**Vocabulary Analysis Script**

In [51]:
from collections import Counter

observe(clean_vocab)

freq = {}

with open("/corpus.txt", "r", encoding="utf-8") as f:
    for line in f:
        pieces = sp.encode(line.strip(), out_type=str)

        for token in pieces:
            if token in freq:
                freq[token] += 1
            else:
                freq[token] = 1

top_10 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 most frequent token pieces:")
for token, count in top_10:
    print(f"{token}: {count}")


OBSERVATION:

Single characters: 100
Subword fragments (2–4 chars): 158
Full words (5+ chars): 41

Examples of single characters: ['հ', 'է', 'Հ', 'կ', 'մ', 'ա', 'լ', 'շ', 'գ', 'դ', 'Ա', 'բ', 'ժ', 'Բ', 'տ', 'Կ', 'Ն', 'Տ', 'զ', 'պ']
Examples of subwords: ['<s>', '</s>', 'ու', 'ան', 'այ', 'եր', 'ար', 'ուն', 'ում', 'ակ', 'ութ', 'ությ', 'են', 'ներ', 'աս', 'Հայ', 'որ', 'ամ', 'ական', 'եւ']
Examples of full words: ['<unk>', 'ություն', 'ությունը', 'աստան', 'Հայաստան', 'կարեւոր', 'Հայաստանում', 'կական', 'ության', 'ավանդ', 'ունեն', 'աշխարհ', 'ությունն', 'ժողով', 'ավանդական', 'անում', 'իրում', 'ակակից', 'ժաման', 'համար']

Top 10 most frequent token pieces:
։: 93
▁է: 53
▁: 41
ն: 35
ան: 33
ի: 33
▁են: 27
ը: 25
ր: 22
ում: 20


**Home Task: Large-Scale Training**

In [30]:
from datasets import load_dataset

dataset = load_dataset("cc100", lang="hy")

print(dataset)

RuntimeError: Dataset scripts are no longer supported, but found cc100.py

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files="https://data.statmt.org/cc-100/hy.txt.xz"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 25355483
    })
})


**Training**

In [41]:
with open("/corpus_large.txt", "w", encoding="utf-8") as f:
    for i, example in enumerate(dataset["train"]):
        if i >= 50000:
            break

        text = example["text"].strip()

        # filter noise
        if len(text) > 20:
            f.write(text + "\n")

# with open("/corpus_large.txt", "r", encoding="utf-8") as f:
#     for i in range(50):
#         print(f.readline())

spm.SentencePieceTrainer.train(
    input="corpus_large.txt",
    model_prefix="hy_bpe_large",
    vocab_size=8000,
    model_type="bpe",
    character_coverage=1.0
)

**Encode & Decode**

In [48]:
sp = spm.SentencePieceProcessor()
sp.load("hy_bpe_large.model")

sentences = [
    "Ես սիրում եմ առավոտյան շուտ արթնանալ, քանի որ այդ ժամանակ օրը ավելի հանգիստ և արդյունավետ է թվում։",
    "Երևանը տարվա բոլոր եղանակներին ունի իր յուրահատուկ գեղեցկությունը, հատկապես գարնանը, երբ քաղաքը ծաղկում է։",
    "Այսօր եղանակը բավականին տաք է, և մարդիկ զբոսնում են այգիներում՝ վայելելով արևոտ օրը։",
    "Գիրք կարդալը ոչ միայն զարգացնում է մտածողությունը, այլ նաև օգնում է հասկանալ տարբեր մշակույթներ և գաղափարներ։",
    "Ուսանողները այս շրջանում շատ զբաղված են, քանի որ նրանք պատրաստվում են իրենց կարևոր քննություններին և նախագծերին։"
]

for s in sentences:
    pieces = sp.encode(s, out_type=str)
    ids = sp.encode(s, out_type=int)
    decoded = sp.decode(ids)

    print("\nSentence:", s)
    print("Pieces:", pieces)
    print("IDs:", ids)
    print("Decoded:", decoded)




Sentence: Ես սիրում եմ առավոտյան շուտ արթնանալ, քանի որ այդ ժամանակ օրը ավելի հանգիստ և արդյունավետ է թվում։
Pieces: ['▁Ես', '▁սիրում', '▁եմ', '▁առավոտյան', '▁շուտ', '▁ար', 'թն', 'անալ', ',', '▁քանի', '▁որ', '▁այդ', '▁ժամանակ', '▁օրը', '▁ավելի', '▁հանգիստ', '▁եւ', '▁արդյունավետ', '▁է', '▁թվում', '։']
IDs: [1154, 2879, 296, 3272, 3091, 96, 2068, 578, 7592, 544, 42, 232, 308, 1387, 405, 3973, 47, 1826, 16, 1285, 7626]
Decoded: Ես սիրում եմ առավոտյան շուտ արթնանալ, քանի որ այդ ժամանակ օրը ավելի հանգիստ եւ արդյունավետ է թվում։

Sentence: Երևանը տարվա բոլոր եղանակներին ունի իր յուրահատուկ գեղեցկությունը, հատկապես գարնանը, երբ քաղաքը ծաղկում է։
Pieces: ['▁Երեւանը', '▁տարվա', '▁բոլոր', '▁եղանակ', 'ներին', '▁ունի', '▁իր', '▁յուրահատուկ', '▁գեղեցկ', 'ությունը', ',', '▁հատկապես', '▁գարն', 'անը', ',', '▁երբ', '▁քաղաքը', '▁ծաղ', 'կում', '▁է', '։']
IDs: [7217, 1192, 410, 2844, 264, 738, 90, 6197, 4648, 129, 7592, 2342, 5861, 353, 7592, 526, 6358, 3468, 7168, 16, 7626]
Decoded: Երեւանը տարվա բոլոր 

**Comparison**

In [47]:
sp_small = spm.SentencePieceProcessor()
sp_small.load("hy_bpe.model")

sp_large = spm.SentencePieceProcessor()
sp_large.load("hy_bpe_large.model")

sentences = [
    "Ես սիրում եմ առավոտյան շուտ արթնանալ, քանի որ այդ ժամանակ օրը ավելի հանգիստ և արդյունավետ է թվում։",
    "Երևանը տարվա բոլոր եղանակներին ունի իր յուրահատուկ գեղեցկությունը, հատկապես գարնանը, երբ քաղաքը ծաղկում է։",
    "Այսօր եղանակը բավականին տաք է, և մարդիկ զբոսնում են այգիներում՝ վայելելով արևոտ օրը։",
    "Գիրք կարդալը ոչ միայն զարգացնում է մտածողությունը, այլ նաև օգնում է հասկանալ տարբեր մշակույթներ և գաղափարներ։",
    "Ուսանողները այս շրջանում շատ զբաղված են, քանի որ նրանք պատրաստվում են իրենց կարևոր քննություններին և նախագծերին։"
]

def avg_tokens(sp, sentences):
    total = 0
    for s in sentences:
        total += len(sp.encode(s))
    return total / len(sentences)

small_avg = avg_tokens(sp_small, sentences)
large_avg = avg_tokens(sp_large, sentences)

print("Average tokens (small model):", small_avg)
print("Average tokens (large model):", large_avg)

Average tokens (small model): 55.6
Average tokens (large model): 22.2


# **Conclusion**

The results show a clear improvement in tokenization efficiency when moving from the small SentencePiece model to the large one. The average number of tokens per sentence decreases significantly from **55.6 tokens (small model)** to **22.2 tokens (large model)**, which is approximately a **2.5× reduction**. This demonstrates that the larger model produces more compact and meaningful representations of text.

This improvement occurs because the large model, trained on a much larger corpus with a bigger vocabulary, is able to learn frequent Armenian words and morphemes as single tokens or longer subword units. As a result, it performs fewer splits and relies less on character-level fragmentation. In contrast, the smaller model tends to over-segment words due to limited exposure to linguistic patterns.

From a computational perspective, fewer tokens per sentence lead to more efficient language modeling. This reduces sequence length, lowers memory usage, and can improve both training and inference speed in downstream NLP tasks. At the same time, the tokenization becomes more aligned with natural language structure, improving semantic representation.

Overall, these results align with theoretical expectations of Byte Pair Encoding (BPE): a smaller model behaves more like a character-level tokenizer, while a larger model shifts toward word-level and morpheme-aware segmentation. This highlights the trade-off between vocabulary size and generalization, and demonstrates how scaling the corpus and vocabulary improves the quality of tokenization for morphologically rich languages such as Armenian.